# REVORA — Phase 2: Recovery Prediction Engine Notebook

This notebook trains, evaluates, and optimizes machine learning models to predict recovery probability on failed payment transactions.

## Key Objectives
1. Fit leak-free pre-recovery feature pipeline on `train.csv` (FAILED transactions only).
2. Train Baseline **Logistic Regression** and Primary Advanced **XGBoost** models.
3. Evaluate statistical metrics (Precision, Recall, F1, ROC-AUC, PR-AUC, Confusion Matrix) on `val.csv`.
4. Optimize decision threshold $\tau^*$ on `val.csv` maximizing Net Recovery Value subject to safety guardrails.
5. Produce feature importance and SHAP interpretability plots.
6. Save persisted model artifacts to `models/`.

In [ ]:
import os
import sys
import pickle
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set project root path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.ml.feature_engineering import FeaturePipeline
from src.ml.baseline import BaselineLogisticRegression
from src.ml.trainer import XGBoostRecoveryModel
from src.ml.evaluator import ModelEvaluator
from src.ml.inference import RecoveryInferenceEngine

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Load Training and Validation Datasets (Test Set Locked)

In [ ]:
train_df = pd.read_csv(os.path.join(project_root, 'data', 'processed', 'train.csv'))
val_df = pd.read_csv(os.path.join(project_root, 'data', 'processed', 'val.csv'))

print(f'Train Total Rows: {len(train_df):,} | Val Total Rows: {len(val_df):,}')

# Feature engineering on FAILED transactions
pipeline = FeaturePipeline()
X_train, y_train, train_failed = pipeline.fit_transform(train_df)
X_val, y_val, val_failed = pipeline.transform(val_df)

print(f'Train Failed Rows: {len(train_failed):,} | Features Count: {X_train.shape[1]}')
print(f'Val Failed Rows:   {len(val_failed):,}')

## 2. Train Models (Baseline Logistic Regression vs Primary Advanced XGBoost)

In [ ]:
# Baseline Logistic Regression
baseline = BaselineLogisticRegression(random_state=42)
baseline.fit(X_train, y_train)
val_baseline_probs = baseline.predict_proba(X_val)

# Primary Advanced XGBoost Classifier
xgb_model = XGBoostRecoveryModel(n_estimators=150, max_depth=4, learning_rate=0.05, random_state=42)
xgb_model.fit(X_train, y_train)
val_xgb_probs = xgb_model.predict_proba(X_val)

## 3. Compare Baseline vs XGBoost on Validation Set

In [ ]:
evaluator = ModelEvaluator(cost_per_retry=10.0)

base_stat = evaluator.evaluate_statistical_metrics(y_val, val_baseline_probs, threshold=0.5)
xgb_stat = evaluator.evaluate_statistical_metrics(y_val, val_xgb_probs, threshold=0.5)

base_biz = evaluator.evaluate_business_metrics(val_failed, val_baseline_probs, threshold=0.5)
xgb_biz = evaluator.evaluate_business_metrics(val_failed, val_xgb_probs, threshold=0.5)

comp_df = pd.DataFrame([
    {'Model': 'Baseline Logistic Regression', 'ROC-AUC': base_stat['roc_auc'], 'PR-AUC': base_stat['pr_auc'], 'F1': base_stat['f1'], 'Net Recovery (INR)': base_biz['net_revenue_recovered']},
    {'Model': 'Primary Advanced XGBoost',    'ROC-AUC': xgb_stat['roc_auc'],  'PR-AUC': xgb_stat['pr_auc'],  'F1': xgb_stat['f1'],  'Net Recovery (INR)': xgb_biz['net_revenue_recovered']}
])
display(comp_df)

## 4. Optimize Decision Threshold $\tau^*$ for Net Recovery Value

In [ ]:
opt_tau, grid_df = evaluator.optimize_threshold(val_failed, val_xgb_probs, max_intervention_rate=0.80)
print(f'Optimal Threshold tau*: {opt_tau:.4f}')

opt_stat = evaluator.evaluate_statistical_metrics(y_val, val_xgb_probs, threshold=opt_tau)
opt_biz = evaluator.evaluate_business_metrics(val_failed, val_xgb_probs, threshold=opt_tau)

print('\n=== Validation Set Performance at Optimal Threshold ===')
for k, v in opt_biz.items():
    print(f'{k:35s}: {v}')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(grid_df['threshold'], grid_df['net_revenue_recovered'], label='Net Recovery Value (INR)', color='#2ecc71', linewidth=2)
ax.axvline(opt_tau, color='red', linestyle='--', label=f'Optimal tau* = {opt_tau:.2f}')
ax.set_title('Threshold vs Net Recovery Value (Validation Set)')
ax.set_xlabel('Probability Threshold (tau)')
ax.set_ylabel('Net Revenue Recovered (INR)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Feature Importance & SHAP Interpretability Analysis

In [ ]:
fi_df = xgb_model.get_feature_importances(pipeline.feature_names_)
print('=== Top 10 Features by Importances ===')
print(fi_df.head(10))

fig, ax = plt.subplots(figsize=(10, 6))
top10 = fi_df.head(10).sort_values(by='importance', ascending=True)
ax.barh(top10['feature'], top10['importance'], color='#3498db')
ax.set_title('Top 10 XGBoost Feature Importances')
ax.set_xlabel('Relative Importance')
plt.tight_layout()
plt.show()